# L10 (6.05)  data exploration of StarCraft dataset 

* StarCraft is an RTS game where players are assigned (based on their skill) to different Leagues. 
* APM is an acronym for Actions Per Minute, i.e. how many actions (sending unit, clicking attack, etc.) given player
performed in one minute

In this notebook we will look at factors affecting league assigned to given player

## New pandas skills
* groupby
* query syntax (no really new capabilities, just easier to use)

## New "general" skills
* calculating correlations
* transforming (usually using log) axis 
* performing quick and dirty (frequential) statistical tests in Python

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns
import numpy as np
import scipy
from scipy import stats

In [ ]:
# Read data, replace missing data by '?'
df = pd.read_csv("data/SkillCraft1_Dataset.csv", na_values=["?"])

In [ ]:
df.head()

### Groupby

In [ ]:
# Groupby returns strange object which exposes few things. 
# If you know SQL this is something that enables you to finish the query (for instance calculate mean in groups)
strange_obj = df.groupby("LeagueIndex")

In [ ]:
m = strange_obj['APM']

In [ ]:
strange_obj['APM'].median()

In [ ]:
# Besides being strange, it is also a list of tuples, where
# each tuple is simply value of column grouped on (or set of columns) and resulting DataFrame
print("Different LeagueIndexes" + str(len(strange_obj)))
dfs = list(strange_obj)
print("LeagueIndex=" + str(dfs[0][0]))
dfs[0][1].head()

In [ ]:
strange_obj['APM'].hist()

In [ ]:
df.groupby("LeagueIndex").size()

In [ ]:
df.groupby("LeagueIndex")['APM'].mean() # Average clicks per minute

In [ ]:
sns.violinplot(x="LeagueIndex", y="Age", data=df, order=[1, 2, 3, 4, 5, 6, 7])

### Minor trick: histogram with set number of bins is less deceiving

In [ ]:
df["LeagueIndex"].hist()

In [ ]:
df["LeagueIndex"].hist(bins=len(df["LeagueIndex"].unique()))

### Correlation

Pearson (linear) correlation captures linear relationships between variables. This can be readily seen
by examining linear regression solution. 

We can quickly visualize correlations in pandas:

In [ ]:
df_some = df.loc[:,"LeagueIndex":"APM"]
df_some.corr()

In [ ]:
sns.heatmap(df_some.corr(), annot=True)

Spearman correlation is not requiring linear relationship and is less sensitive to outliers

In [ ]:
sns.heatmap(df_some.corr(method='spearman'), annot=True)

In [ ]:
# mo?emy te??wykre?li? r??nic??ich kwadrat?w
sns.heatmap(df_some.corr(method='spearman')**2 - df_some.corr(method='pearson')**2)

### Statistical tests

It is useful to know how to perform basic statistical tests (one reason being people often ask for such).

* Simple pearson correlation test. It just assesses if linear relationship between 2 random variables could
be by chance

In [ ]:
df_scaled = df.query("HoursPerWeek > 0").copy() # Will be needed for log10
df_scaled['APM_log10'] = np.log10(df_scaled['APM'])
df_scaled['HoursPerWeek_log10'] = np.log10(df_scaled['HoursPerWeek'])

In [ ]:
## Pearson correlation using scipy
# Very low p-value (easily below magic 0.05), so we reject null hypothesis
scipy.stats.spearmanr(df_scaled['APM_log10'], df_scaled['HoursPerWeek_log10']) 

In [ ]:
## Pearson correlation visually (legend in the corner)
sns.jointplot(x="APM_log10", y="HoursPerWeek_log10",
              data=df_scaled, kind="kde")

 * ANOVA (ANalysis Of VAriances) can be for instance used to assess if two groups have significant differences

In [ ]:
# Create dict league_index -> APM (actions per minute)
data = {}
# Note: would be slightly shorter using groupby
for league_index in range(1, 8):
    # Note: same as
    # df[df['LeagueIndex'] == league_index]
    data[league_index] = df.query("LeagueIndex=={}".format(league_index))['APM']
    # Convert to numpy array
    data[league_index] = np.array(data[league_index])

In [ ]:
plt.hist(data.values()[0], alpha=0.5, label=data.keys()[0:1])
plt.hist(data.values()[1], alpha=0.5, label=data.keys()[1:2])
plt.legend()

In [ ]:
test_result = scipy.stats.f_oneway(data.values()[0], data.values()[1])
print("pvalue=" + str(test_result.pvalue)) # Test passed, 2e-11 for chance result

## Exercise 1 [are players' ages significantly different across different leagues?, 2 points]

* Select all data, where LeagueIndex is in [1,3,6] (you can use df.query() syntax).

* Include histograms of ages for each LeagueIndex (3 plots), make sure histograms include legend

* Include boxplot of ages for each LeagueIndex

* What is the mean and std of age for each LeagueIndex? (you can use groupby)

* Perform oneway ANOVA test

## Exercise 2 [10000 hours to master any subject?, 2 points]

* Plot boxplot with x="LeagueIndex", y="TotalHours" (see previous lecture).

You should get something like this: <img width=300, src="img/10/ex2.png">

* There is a popular saying that it takes 10k hours to master any skill. Is the single most important factor affecting LeagueIndex is spent time playing StarCraft? Include plots/statistics that confirm your conclusions.

# Exercise 3 [Produce cool plot for your company meeting, 1 point]

* Add new columns 'Age_log10', 'TotalHours_log10', 'HoursPerWeek_log10', 'APM_log10', which are log transformed corrsponding columns in the original DataFrame
* Using seaborn.pairplot produce plot similar to this <img src="img/10/ex3.png">